# Parkinson's Disease Voice Screening & Clinical Decision Support Platform
## Notebook 04: Model Training with Mixed Precision, Class-Weighting & Drive Checkpointing

> **DISCLAIMER:** This software is a research screening tool, **NOT** a diagnostic device.

### Instructions for Running on Google Colab (Free T4 GPU):
1. **Enable GPU Runtime:**
   - In Google Colab, go to menu: **Runtime** -> **Change runtime type** -> select **T4 GPU** -> click **Save**.
2. **Mount Google Drive (Optional but Recommended):**
   - Run Cell 1 to mount Google Drive. This ensures all model checkpoints (`best_model.pt` and `latest.pt`) are continuously backed up to your Drive, protecting against Colab session timeouts.
3. **Repository Files:**
   - Upload the project folder to your Drive (or clone your repo) and set the path in Cell 1.
4. **Key Features:**
   - **T4 Mixed Precision:** Uses `torch.amp.autocast('cuda')` and `GradScaler` for ultra-fast training and minimal VRAM usage.
   - **Class Imbalance Mitigation:** Dynamically weights the minority class using `pos_weight` in `BCEWithLogitsLoss`.
   - **Resumable Training:** If Colab disconnects, re-running Cell 7 automatically resumes from `latest.pt` without losing progress.
   - **Early Stopping:** Monitored on Validation ROC-AUC with patience = 10 epochs.


In [1]:
# Cell 1: Environment setup, dependencies, and optional Google Drive mount
import os
import sys
import subprocess
from pathlib import Path

# Check if running in Google Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab environment.")
    # Mount Google Drive for persistent checkpoint storage
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_CHECKPOINT_DIR = Path("/content/drive/MyDrive/parkinsons_platform/checkpoints")
        DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Google Drive mounted. Checkpoints will be backed up to: {DRIVE_CHECKPOINT_DIR}")
    except Exception as e:
        print(f"Drive mount skipped or failed: {e}. Using local storage.")
        DRIVE_CHECKPOINT_DIR = None

    # Install required packages in Colab runtime
    required_packages = ["transformers", "torch", "torchaudio", "pandas", "numpy", "scikit-learn", "matplotlib", "tqdm"]
    for pkg in required_packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
else:
    print("Running in standard local / remote environment.")
    DRIVE_CHECKPOINT_DIR = None

import json
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from transformers import get_cosine_schedule_with_warmup

# Resolve repository root
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT.resolve()}")
print("Dependencies loaded successfully.")


In [2]:
# Cell 2: Deterministic seeding and GPU device selection
TRAIN_CONFIG_PATH = PROJECT_ROOT / "config" / "training_config.json"
MODEL_CONFIG_PATH = PROJECT_ROOT / "ml" / "model_def" / "model_config.json"
METADATA_PATH = PROJECT_ROOT / "data" / "processed" / "metadata.csv"

with open(TRAIN_CONFIG_PATH, "r", encoding="utf-8") as f:
    train_cfg = json.load(f)

# Reproducibility seed
SEED = train_cfg.get("seed", 42)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"✓ GPU Acceleration Active: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✓ Apple Silicon MPS Active.")
else:
    device = torch.device("cpu")
    print("Notice: Running on CPU.")

print(f"Fixed Seed: {SEED}")


In [3]:
# Cell 3: Load ModelConfig and import ParkinsonsVoiceClassifier
from ml.model_def.model import ModelConfig, ParkinsonsVoiceClassifier

model_cfg = ModelConfig.from_json(MODEL_CONFIG_PATH)
print("=== MODEL ARCHITECTURE CONFIGURATION ===")
print(f"  Input Feature Dim:    {model_cfg.feature_dim}")
print(f"  Temporal Sequence T:  {model_cfg.temporal_frames}")
print(f"  Stem Channels:        {model_cfg.stem_channels}")
print(f"  Stem Depths:          {model_cfg.stem_depths}")
print(f"  Transformer Dim:      {model_cfg.transformer_dim}")
print(f"  Transformer Layers:   {model_cfg.transformer_layers}")
print(f"  Transformer Heads:    {model_cfg.transformer_heads}")
print(f"  Dropout Rate:         {model_cfg.dropout}")
print("=========================================")

model = ParkinsonsVoiceClassifier(model_cfg).to(device)
param_info = model.count_parameters()
print(f"Model instantiated with {param_info['total']:,} parameters ({param_info['total']/1e6:.2f}M).")


In [4]:
# Cell 4: PyTorch Dataset and DataLoaders
class VoiceFeatureDataset(Dataset):
    """Dataset loading cached (199, 768) WavLM frame features from disk."""

    def __init__(self, df: pd.DataFrame, root_dir: Path):
        self.records = df.to_dict("records")
        self.root_dir = root_dir

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        item = self.records[idx]
        feat_path = self.root_dir / item["feature_path"]
        features = np.load(str(feat_path))  # shape: (199, 768)
        label = float(item["label"])        # 0.0 or 1.0
        return torch.tensor(features, dtype=torch.float32), torch.tensor([label], dtype=torch.float32)

meta_df = pd.read_csv(METADATA_PATH)
train_df = meta_df[meta_df["split"] == "train"].reset_index(drop=True)
val_df = meta_df[meta_df["split"] == "val"].reset_index(drop=True)

# Batch size configuration (start at 16; reduce to 8 if GPU memory is constrained)
BATCH_SIZE = train_cfg.get("batch_size", 16)
NUM_WORKERS = train_cfg.get("num_workers", 2) if device.type == "cuda" else 0

train_dataset = VoiceFeatureDataset(train_df, PROJECT_ROOT)
val_dataset = VoiceFeatureDataset(val_df, PROJECT_ROOT)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print(f"Train samples: {len(train_dataset)} ({len(train_loader)} batches of {BATCH_SIZE})")
print(f"Val samples:   {len(val_dataset)} ({len(val_loader)} batches of {BATCH_SIZE})")


In [5]:
# Cell 5: Compute class weights to address train split imbalance
# From Prompt 2 EDA: 283 Healthy (label 0) vs 301 Parkinson's (label 1)
n_healthy = (train_df["label"] == 0).sum()
n_pd = (train_df["label"] == 1).sum()

pos_weight_value = n_healthy / n_pd
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)

print("=== CLASS IMBALANCE MITIGATION ===")
print(f"  Train Healthy (Negative):    {n_healthy}")
print(f"  Train Parkinson's (Positive): {n_pd}")
print(f"  Computed pos_weight:         {pos_weight_value:.4f}")
print("==================================")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


In [6]:
# Cell 6: AdamW optimizer, cosine decay with linear warmup, and GradScaler
LEARNING_RATE = train_cfg.get("lr", 3e-4)
WEIGHT_DECAY = train_cfg.get("weight_decay", 0.01)
MAX_EPOCHS = train_cfg.get("max_epochs", 35)
WARMUP_RATIO = train_cfg.get("warmup_ratio", 0.05)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

total_steps = len(train_loader) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# Mixed precision scaler (GPU only)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

print(f"Optimizer: AdamW (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"Scheduler: Cosine decay with {warmup_steps} warmup steps ({total_steps} total steps)")
print(f"Mixed Precision (AMP): {'Enabled' if device.type == 'cuda' else 'Disabled'}")


In [7]:
# Cell 7: Resumable checkpoint check
CKPT_DIR = PROJECT_ROOT / train_cfg.get("checkpoint_dir", "models/checkpoints")
ART_DIR = PROJECT_ROOT / train_cfg.get("artifact_dir", "models/artifact")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

LATEST_CKPT = CKPT_DIR / "latest.pt"
BEST_CKPT = CKPT_DIR / "best_model.pt"
LOG_CSV_PATH = PROJECT_ROOT / train_cfg.get("log_csv_path", "training/notebooks/training_log.csv")

start_epoch = 1
best_val_auc = -1.0
training_logs = []

if LATEST_CKPT.exists():
    print(f"Found existing checkpoint at {LATEST_CKPT}.")
    print("Loading saved state for seamless resumption...")
    checkpoint = torch.load(LATEST_CKPT, map_location=device)
    start_epoch = checkpoint["epoch"] + 1
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    best_val_auc = checkpoint.get("best_val_auc", -1.0)
    if LOG_CSV_PATH.exists():
        training_logs = pd.read_csv(LOG_CSV_PATH).to_dict("records")
    print(f"✓ Resumed from Epoch {start_epoch - 1}. Current best Val ROC-AUC: {best_val_auc:.4f}")
else:
    print("No previous checkpoint found. Starting training from Epoch 1.")


In [8]:
# Cell 8: Training and Validation Loop with Early Stopping
PATIENCE = train_cfg.get("patience", 10)
patience_counter = 0

print(f"Training for {MAX_EPOCHS} epochs with patience = {PATIENCE}...")
print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>8} | {'Val Acc':>7} | {'Val Prec':>8} | {'Val Rec':>7} | {'Val F1':>7} | {'Val AUC':>8} | {'LR':>9}")
print("-" * 88)

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    model.train()
    running_train_loss = 0.0

    for features, targets in train_loader:
        features = features.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        # Mixed precision forward pass
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits, _ = model(features)
            loss = criterion(logits, targets)

        if device.type == "cuda":
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        scheduler.step()
        running_train_loss += loss.item() * features.size(0)

    train_loss = running_train_loss / len(train_dataset)

    # Validation pass
    model.eval()
    val_loss_sum = 0.0
    val_probs_all = []
    val_preds_all = []
    val_targets_all = []

    with torch.no_grad():
        for features, targets in val_loader:
            features = features.to(device)
            targets = targets.to(device)

            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                logits, _ = model(features)
                v_loss = criterion(logits, targets)

            val_loss_sum += v_loss.item() * features.size(0)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
            preds = (probs >= 0.5).astype(int)

            val_probs_all.extend(probs)
            val_preds_all.extend(preds)
            val_targets_all.extend(targets.cpu().numpy().flatten().astype(int))

    val_loss = val_loss_sum / len(val_dataset)
    val_acc = accuracy_score(val_targets_all, val_preds_all)
    val_prec = precision_score(val_targets_all, val_preds_all, zero_division=0)
    val_rec = recall_score(val_targets_all, val_preds_all, zero_division=0)
    val_f1 = f1_score(val_targets_all, val_preds_all, zero_division=0)
    val_auc = roc_auc_score(val_targets_all, val_probs_all)
    current_lr = scheduler.get_last_lr()[0]

    log_entry = {
        "epoch": epoch,
        "train_loss": round(train_loss, 4),
        "val_loss": round(val_loss, 4),
        "val_acc": round(val_acc, 4),
        "val_prec": round(val_prec, 4),
        "val_rec": round(val_rec, 4),
        "val_f1": round(val_f1, 4),
        "val_roc_auc": round(val_auc, 4),
        "lr": round(current_lr, 7),
    }
    training_logs.append(log_entry)

    # Save latest checkpoint every epoch for seamless recovery
    ckpt_state = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_auc": best_val_auc,
        "val_roc_auc": val_auc,
        "val_loss": val_loss,
        "model_config": model_cfg.to_dict(),
        "training_config": train_cfg,
    }
    torch.save(ckpt_state, str(LATEST_CKPT))
    if DRIVE_CHECKPOINT_DIR:
        torch.save(ckpt_state, str(DRIVE_CHECKPOINT_DIR / "latest.pt"))

    # Check for improvement
    is_best = False
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_counter = 0
        is_best = True
        torch.save(ckpt_state, str(BEST_CKPT))
        torch.save(ckpt_state, str(ART_DIR / "best_model.pt"))
        if DRIVE_CHECKPOINT_DIR:
            torch.save(ckpt_state, str(DRIVE_CHECKPOINT_DIR / "best_model.pt"))
    else:
        patience_counter += 1

    marker = " [★ BEST]" if is_best else ""
    print(f"{epoch:5d} | {train_loss:10.4f} | {val_loss:8.4f} | {val_acc:7.4f} | {val_prec:8.4f} | {val_rec:7.4f} | {val_f1:7.4f} | {val_auc:8.4f} | {current_lr:9.2e}{marker}")

    # Persist log CSV
    pd.DataFrame(training_logs).to_csv(LOG_CSV_PATH, index=False)

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping triggered: No validation ROC-AUC improvement for {PATIENCE} epochs.")
        break

print(f"\nTraining complete! Best Validation ROC-AUC achieved: {best_val_auc:.4f}")


In [9]:
# Cell 9: Plot training loss and validation metric progression
log_df = pd.DataFrame(training_logs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(log_df["epoch"], log_df["train_loss"], label="Train Loss", color="royalblue", lw=2)
ax1.plot(log_df["epoch"], log_df["val_loss"], label="Val Loss", color="coral", lw=2, linestyle="--")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Binary Cross Entropy Loss")
ax1.set_title("Training & Validation Loss Progression")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Performance metrics
ax2.plot(log_df["epoch"], log_df["val_roc_auc"], label="Val ROC-AUC", color="darkgreen", lw=2)
ax2.plot(log_df["epoch"], log_df["val_f1"], label="Val F1-Score", color="purple", lw=2, linestyle="--")
ax2.plot(log_df["epoch"], log_df["val_acc"], label="Val Accuracy", color="orange", lw=1.5, linestyle=":")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score [0.0 - 1.0]")
ax2.set_title("Validation Metric Progression")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [10]:
# Cell 10: Verify saved checkpoint integrity
print("=== SAVED CHECKPOINT AUDIT ===")

for ckpt_path in [BEST_CKPT, LATEST_CKPT, ART_DIR / "best_model.pt"]:
    assert ckpt_path.exists(), f"Checkpoint missing: {ckpt_path}"
    state = torch.load(ckpt_path, map_location="cpu")
    assert "model_state_dict" in state, f"Corrupt checkpoint: {ckpt_path}"
    print(f"✓ Verified {ckpt_path.name:15s} (Epoch {state['epoch']}, Val AUC: {state['val_roc_auc']:.4f})")

print("\nModel ready for Prompt 5 evaluation on held-out test split and MDVR-KCL.")
